# Urea Reaction Path LP: GAMSPy Solution

CHBE444: Design I, Prof. Ganesh Sriram (gsriram@umd.edu), University of Maryland.

*This LP was introduced in a past lecture.*

In [3]:
import gamspy as gams
import numpy as np
import pandas as pd


def print_records(vars):
    for k in np.arange(0, len(vars)):
        print(vars[k].name, ':')
        display(vars[k].records)
        print()


m = gams.Container()

i = gams.Set(
    container=m,
    name='i',
    description='compound'
)

i.setRecords(['CH4', 'CO', 'CO2',
              'H2', 'N2', 'NH3', 'urea'])  # compounds except H2O
print_records([i])

j = gams.Set(
    container=m,
    name='j',
    description='reaction'
)

j.setRecords(['rxn0', 'rxn1', 'rxn2', 'rxn3'])  # reactions (x0, x1, x2, x3)
print_records([j])

yields = gams.Parameter(
    container=m,
    name='yields',
    domain=j,
    description='contribution of reaction weights to yield (objective)'
)

yields.setRecords([('rxn0', 0),
                   ('rxn1', 0),
                   ('rxn2', 0),
                   ('rxn3', 0.5)])
print_records([yields])

# The throughput constraints are split into two parts:
# those with throughput <= 0 and those with throughputs >= 0
# Methane has a (basis) throughput of exactly -1, so it is
# included in both sets of constraints

stoich_matrix_lesser = pd.DataFrame(
    [
        ['CH4', 'rxn0', -1],
        ['CO', 'rxn0', 1],
        ['CO', 'rxn1', -1],
        ['CO2', 'rxn1', 1],
        ['CO2', 'rxn3', -0.5],
        ['N2', 'rxn2', -1]
    ], columns=['compound', 'reaction', 'coefficient']
).set_index(['compound', 'reaction'])

stoich_lesser = gams.Parameter(
    container=m,
    name='stoich_lesser',
    domain=[i, j],
    description='stoichiometric coefficient of compound i in reaction j',
    records=stoich_matrix_lesser.reset_index()
)
print_records([stoich_lesser])

stoich_matrix_greater = pd.DataFrame(
    [
        ['CH4', 'rxn0', -1],
        ['H2', 'rxn0', 3],
        ['H2', 'rxn1', 1],
        ['H2', 'rxn2', - 3],
        ['NH3', 'rxn2', 2],
        ['NH3', 'rxn3', -1],
        ['urea', 'rxn3', 0.5]
    ], columns=['compound', 'reaction', 'coefficient']
).set_index(['compound', 'reaction'])

stoich_greater = gams.Parameter(
    container=m,
    name='stoich_greater',
    domain=[i, j],
    description='stoichiometric coefficient of compound i in reaction j',
    records=stoich_matrix_greater.reset_index()
)
print_records([stoich_greater])

x = gams.Variable(
    container=m,
    name='x',
    domain=j,
    type='Positive',
    description='weight of reaction j in process'
)

throughput_lesser = gams.Parameter(
    container=m,
    name='throughput_lesser',
    domain=i,
    description='throughput of compound i in process when <= 0'
)

throughput_lesser.setRecords([('CH4', -1),
                              ('CO', 0),
                              ('CO2', 0),
                              ('N2', 0)])
print_records([throughput_lesser])

throughput_greater = gams.Parameter(
    container=m,
    name='throughput_lesser',
    domain=i,
    description='throughput of compound i in process when >= 0'
)

throughput_greater.setRecords([('CH4', -1),
                               ('H2', 0),
                               ('NH3', 0),
                               ('urea', 0)])

print_records([throughput_greater])

throughput_lesser_const = gams.Equation(
    container=m,
    name='throughput_lesser_const',
    domain=i,
    description='throughput of compound i when <= 0'
)

throughput_greater_const = gams.Equation(
    container=m,
    name='throughput_greater_const',
    domain=i,
    description='throughput of compound i when >= 0'
)

throughput_lesser_const[i] = gams.Sum(
    j, stoich_lesser[i, j] * x[j]) <= throughput_lesser[i]

throughput_greater_const[i] = gams.Sum(
    j, stoich_greater[i, j] * x[j]) >= throughput_greater[i]


obj = gams.Sum(j, yields[j] * x[j])

problem = gams.Model(
    container=m,
    name='urea',
    equations=m.getEquations(),
    problem='LP',
    sense=gams.Sense.MAX,
    objective=obj
)

problem.solve(
    output=None,
    options=gams.Options(equation_listing_limit=10,
                         variable_listing_limit=10)
)

print(problem.getEquationListing())
print()
print('Optimal values of x:')
print(x.records)
print()
print('Optimal value of z:')
print(problem.objective_value)

i :


,uni,element_text
0,CH4,
1,CO,
2,CO2,
3,H2,
4,N2,
5,NH3,
6,urea,



j :


,uni,element_text
0,rxn0,
1,rxn1,
2,rxn2,
3,rxn3,



yields :


...


throughput_lesser :


,i,value
0,CH4,-1.0
1,CO,0.0
2,CO2,0.0
3,N2,0.0



throughput_lesser :


,i,value
0,CH4,-1.0
1,H2,0.0
2,NH3,0.0
3,urea,0.0



throughput_lesser_const(CH4)..  - x(rxn0) =L= -1 ; (LHS = 0, INFES = 1 ****)
throughput_lesser_const(CO)..  x(rxn0) - x(rxn1) =L= 0 ; (LHS = 0)
throughput_lesser_const(CO2)..  x(rxn1) - 0.5*x(rxn3) =L= 0 ; (LHS = 0)
throughput_lesser_const(N2)..  - x(rxn2) =L= 0 ; (LHS = 0)
throughput_greater_const(CH4)..  - x(rxn0) =G= -1 ; (LHS = 0)
throughput_greater_const(H2)..  3*x(rxn0) + x(rxn1) - 3*x(rxn2) =G= 0 ; (LHS = 0)
throughput_greater_const(NH3)..  2*x(rxn2) - x(rxn3) =G= 0 ; (LHS = 0)
throughput_greater_const(urea)..  0.5*x(rxn3) =G= 0 ; (LHS = 0)
urea_objective..  0.5*x(rxn3) - urea_objective_variable =E= 0 ; (LHS = 0)

Optimal values of x:
      j  level  marginal  lower  upper  scale
0  rxn0    1.0       0.0    0.0    inf    1.0
1  rxn1    1.5       0.0    0.0    inf    1.0
2  rxn2    1.5       0.0    0.0    inf    1.0
3  rxn3    3.0       0.0    0.0    inf    1.0

Optimal value of z:
1.4999999999999998
